# Bloco 04 — Window Functions e Transformações Avançadas

**Trilha Databricks · Aula 08 · Jornada de Dados**

Neste bloco, vamos dominar **Window Functions** em PySpark — o conceito que permite fazer cálculos analíticos mantendo todas as linhas do DataFrame.

Você já conhece window functions em SQL (viu no Workshop de Modelagem). Agora vai aprender a sintaxe PySpark com `Window.partitionBy().orderBy()`.

**O que vamos construir:**
- Ranking de clientes por receita
- Receitas acumuladas mensais com YTD (Relatório 5)
- Segmentação de clientes com NTILE (Relatório 6)
- Clientes para marketing (Relatório 7)
- Tabelas pivotadas

## Setup

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "northwind"
SCHEMA = "bronze"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Carregar tabelas
orders = spark.table("orders")
order_details = spark.table("order_details")
customers = spark.table("customers")
products = spark.table("products")
categories = spark.table("categories")

# Preparar base de receita — fórmula central do Northwind
# receita = unit_price * quantity * (1.0 - discount)
od_receita = order_details.withColumn(
    "receita", F.col("unit_price") * F.col("quantity") * (F.lit(1.0) - F.col("discount"))
)

print("Tabelas carregadas com sucesso!")
print(f"order_details com receita: {od_receita.count()} registros")

---

## 1. Window Functions — Conceito

### O que é uma Window Function?

Uma **window function** realiza cálculos sobre um conjunto de linhas relacionadas à linha atual, **sem reduzir o número de linhas** do DataFrame.

### Diferença fundamental: `groupBy` vs `Window`

| Característica | `groupBy().agg()` | Window Function |
|---|---|---|
| Linhas de saída | **Reduz** (1 linha por grupo) | **Mantém** todas as linhas |
| Uso típico | Totais, médias, contagens | Rankings, acumulados, comparações |
| Sintaxe PySpark | `df.groupBy().agg()` | `F.func().over(window)` |

### Sintaxe PySpark

```python
from pyspark.sql.window import Window

# Definir a janela
w = Window.partitionBy("coluna_grupo").orderBy("coluna_ordem")

# Aplicar a função sobre a janela
df.withColumn("nova_coluna", F.rank().over(w))
```

- **`partitionBy()`** — define os grupos (equivalente ao PARTITION BY do SQL)
- **`orderBy()`** — define a ordenação dentro de cada grupo
- **`.over(w)`** — aplica a função sobre a janela definida

---

## 2. Funções de Ranking

As três funções de ranking mais usadas:

| Função | Comportamento com empates |
|---|---|
| `row_number()` | Sempre sequencial (1, 2, 3, 4...) — desempata arbitrariamente |
| `rank()` | Pula posições em empate (1, 2, 2, 4...) |
| `dense_rank()` | Não pula posições (1, 2, 2, 3...) |

### Exemplo: Ranking de clientes por receita total

In [ ]:
# Primeiro, calcular a receita total por cliente
receita_cliente = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .groupBy("customer_id", "company_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
)

print(f"Total de clientes: {receita_cliente.count()}")
receita_cliente.orderBy(F.col("total").desc()).show(10, truncate=False)

In [ ]:
# Aplicar as três funções de ranking
w = Window.orderBy(F.col("total").desc())

ranking = (
    receita_cliente
    .withColumn("row_number", F.row_number().over(w))
    .withColumn("rank", F.rank().over(w))
    .withColumn("dense_rank", F.dense_rank().over(w))
)

ranking.show(15, truncate=False)

**Observe a diferença:**
- `row_number` sempre incrementa: 1, 2, 3, 4...
- `rank` pula posições quando há empate
- `dense_rank` não pula posições

**Equivalente SQL:**
```sql
SELECT 
    customer_id, company_name, total,
    ROW_NUMBER() OVER (ORDER BY total DESC) AS row_number,
    RANK() OVER (ORDER BY total DESC) AS rank,
    DENSE_RANK() OVER (ORDER BY total DESC) AS dense_rank
FROM receita_por_cliente
```

### Ranking com partição: Top produto por categoria

In [ ]:
# Receita por produto e categoria
receita_produto = (
    od_receita
    .join(products, "product_id")
    .join(categories, "category_id")
    .groupBy("category_name", "product_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
)

# Ranking DENTRO de cada categoria
w_cat = Window.partitionBy("category_name").orderBy(F.col("total").desc())

top_por_categoria = (
    receita_produto
    .withColumn("rank_na_categoria", F.rank().over(w_cat))
    .filter(F.col("rank_na_categoria") <= 3)  # Top 3 por categoria
    .orderBy("category_name", "rank_na_categoria")
)

top_por_categoria.show(30, truncate=False)

---

## 3. Funções Analíticas: `lag()`, `lead()` e Soma Acumulada

### `lag()` e `lead()`

- **`lag(col, n)`** — retorna o valor de `n` linhas **anteriores** (padrão: 1)
- **`lead(col, n)`** — retorna o valor de `n` linhas **posteriores** (padrão: 1)

Ideal para calcular diferenças entre períodos (mês anterior, crescimento, etc.)

### `sum().over()` — Soma Acumulada

Quando usamos `F.sum(col).over(window)` com uma janela ordenada, obtemos uma **soma acumulada** (running total).

### Exemplo: Receita mensal com lag

In [ ]:
# Calcular receita mensal
receita_mensal = (
    od_receita
    .join(orders, "order_id")
    .withColumn("ano", F.year("order_date"))
    .withColumn("mes", F.month("order_date"))
    .groupBy("ano", "mes")
    .agg(F.round(F.sum("receita"), 2).alias("receita_mensal"))
    .orderBy("ano", "mes")
)

receita_mensal.show(30)

In [ ]:
# Usar lag para comparar com mês anterior
w_lag = Window.partitionBy("ano").orderBy("mes")

receita_com_lag = (
    receita_mensal
    .withColumn("receita_mes_anterior", F.lag("receita_mensal").over(w_lag))
    .withColumn("diferenca", 
        F.round(F.col("receita_mensal") - F.lag("receita_mensal").over(w_lag), 2))
)

receita_com_lag.show(30)

**Observe:** O primeiro mês de cada ano tem `null` no lag — porque não existe mês anterior dentro daquela partição.

**Equivalente SQL:**
```sql
SELECT 
    ano, mes, receita_mensal,
    LAG(receita_mensal) OVER (PARTITION BY ano ORDER BY mes) AS receita_mes_anterior
FROM receita_mensal
```

---

## Relatório 5 — Receitas Acumuladas Mensais com YTD

**Equivalente SQL:** `view_receitas_acumuladas`

Este relatório combina:
- **Receita mensal** — total de cada mês
- **Receita YTD** (Year-To-Date) — soma acumulada no ano via `SUM().over()`
- **Diferença mensal** — comparação com mês anterior via `LAG()`
- **Percentual de mudança** — crescimento/queda percentual

In [ ]:
# Receita mensal (base)
receita_mensal = (
    od_receita
    .join(orders, "order_id")
    .withColumn("ano", F.year("order_date"))
    .withColumn("mes", F.month("order_date"))
    .groupBy("ano", "mes")
    .agg(F.round(F.sum("receita"), 2).alias("receita_mensal"))
    .orderBy("ano", "mes")
)

# Definir janelas
w_ytd = Window.partitionBy("ano").orderBy("mes")
w_lag = Window.partitionBy("ano").orderBy("mes")

# Construir relatório completo
receitas_acumuladas = (
    receita_mensal
    .withColumn("receita_ytd", F.round(F.sum("receita_mensal").over(w_ytd), 2))
    .withColumn("diferenca_mensal",
        F.round(F.col("receita_mensal") - F.lag("receita_mensal").over(w_lag), 2))
    .withColumn("percentual_mudanca",
        F.round(
            (F.col("receita_mensal") - F.lag("receita_mensal").over(w_lag))
            / F.lag("receita_mensal").over(w_lag) * 100, 2
        ))
)

receitas_acumuladas.show(30)

### Equivalente SQL

```sql
-- view_receitas_acumuladas
SELECT
    ano,
    mes,
    receita_mensal,
    SUM(receita_mensal) OVER (PARTITION BY ano ORDER BY mes) AS receita_ytd,
    receita_mensal - LAG(receita_mensal) OVER (PARTITION BY ano ORDER BY mes) AS diferenca_mensal,
    ROUND(
        (receita_mensal - LAG(receita_mensal) OVER (PARTITION BY ano ORDER BY mes))
        / LAG(receita_mensal) OVER (PARTITION BY ano ORDER BY mes) * 100, 2
    ) AS percentual_mudanca
FROM (
    SELECT
        YEAR(o.order_date) AS ano,
        MONTH(o.order_date) AS mes,
        ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS receita_mensal
    FROM order_details od
    JOIN orders o ON od.order_id = o.order_id
    GROUP BY YEAR(o.order_date), MONTH(o.order_date)
)
ORDER BY ano, mes
```

**Observe:** Em PySpark, o encadeamento de `withColumn` torna o código mais legível que o SQL aninhado.

### Análise do resultado

Com este relatório, conseguimos responder:
- Qual mês teve maior crescimento percentual?
- Em que ponto do ano a receita acumulada ultrapassa determinado valor?
- Há sazonalidade nas vendas?

In [ ]:
# Qual mês teve o maior crescimento percentual?
receitas_acumuladas.orderBy(F.col("percentual_mudanca").desc()).show(5)

---

## 4. Segmentação com NTILE

### O que é `ntile(n)`?

A função `ntile(n)` divide as linhas em **n grupos de tamanho aproximadamente igual**, com base na ordenação definida.

- `ntile(4)` — quartis (4 grupos de 25%)
- `ntile(5)` — quintis (5 grupos de 20%)
- `ntile(10)` — decis (10 grupos de 10%)

O grupo **1** contém os maiores valores (se ordenado `desc`), e o grupo **n** contém os menores.

### Uso prático

Segmentar clientes por valor:
- **Grupo 1-2:** Clientes de alto valor (VIP)
- **Grupo 3:** Clientes intermediários
- **Grupo 4-5:** Clientes de baixo valor (candidatos a campanhas de marketing)

---

## Relatório 6 — Segmentação de Clientes

**Equivalente SQL:** `view_total_revenues_per_customer_group`

Dividir os clientes em 5 grupos por receita total, usando `ntile(5)`.

In [ ]:
# Receita total por cliente
receita_cliente = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .groupBy("company_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
)

# Segmentar em 5 grupos com NTILE
w_ntile = Window.orderBy(F.col("total").desc())

segmentacao = receita_cliente.withColumn("grupo", F.ntile(5).over(w_ntile))

segmentacao.orderBy(F.col("total").desc()).show(30, truncate=False)

In [ ]:
# Resumo por grupo: quantos clientes e receita total de cada grupo
segmentacao.groupBy("grupo").agg(
    F.count("company_name").alias("qtd_clientes"),
    F.round(F.sum("total"), 2).alias("receita_grupo"),
    F.round(F.avg("total"), 2).alias("media_por_cliente"),
    F.round(F.min("total"), 2).alias("min_receita"),
    F.round(F.max("total"), 2).alias("max_receita")
).orderBy("grupo").show()

**Equivalente SQL:**
```sql
-- view_total_revenues_per_customer_group
SELECT
    company_name,
    ROUND(SUM(od.unit_price * od.quantity * (1.0 - od.discount)), 2) AS total,
    NTILE(5) OVER (ORDER BY SUM(od.unit_price * od.quantity * (1.0 - od.discount)) DESC) AS grupo
FROM order_details od
JOIN orders o ON od.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY company_name
ORDER BY total DESC
```

---

## Relatório 7 — Clientes para Marketing

**Equivalente SQL:** `clients_to_marketing`

Filtrar os clientes dos grupos 3, 4 e 5 (menor valor) para direcionar campanhas de marketing e reativação.

In [ ]:
# Filtrar grupos de baixo valor (grupos >= 3)
clientes_marketing = segmentacao.filter(F.col("grupo") >= 3)

print(f"Total de clientes para marketing: {clientes_marketing.count()}")
clientes_marketing.orderBy(F.col("total").desc()).show(truncate=False)

**Equivalente SQL:**
```sql
-- clients_to_marketing
SELECT company_name, total, grupo
FROM view_total_revenues_per_customer_group
WHERE grupo >= 3
ORDER BY total DESC
```

**Observe como em PySpark encadeamos:** calculamos a segmentação e já filtramos no mesmo fluxo, sem precisar criar uma view intermediária.

---

## 5. Pivot Tables

### O que é Pivot?

A operação `pivot()` transforma **valores de uma coluna em novas colunas**. É o equivalente PySpark da tabela dinâmica do Excel.

### Sintaxe

```python
df.groupBy("coluna_fixa").pivot("coluna_que_vira_header").agg(funcao_agregacao)
```

**Exemplo conceitual:**

Antes (formato longo):
| categoria | ano | receita |
|---|---|---|
| Beverages | 1996 | 1000 |
| Beverages | 1997 | 2000 |

Depois do pivot (formato largo):
| categoria | 1996 | 1997 |
|---|---|---|
| Beverages | 1000 | 2000 |

### Exemplo: Receita por categoria pivotada por ano

In [ ]:
# Receita por categoria e ano
receita_cat_ano = (
    od_receita
    .join(orders, "order_id")
    .join(products, "product_id")
    .join(categories, "category_id")
    .withColumn("ano", F.year("order_date"))
    .groupBy("category_name")
    .pivot("ano")
    .agg(F.round(F.sum("receita"), 2))
    .orderBy("category_name")
)

receita_cat_ano.show(truncate=False)

**Dica de performance:** Se você sabe quais são os valores da coluna pivotada, passe-os como lista para evitar que o Spark precise fazer um scan extra:

```python
# Mais eficiente — especifica os valores do pivot
.pivot("ano", [1996, 1997, 1998])
```

In [ ]:
# Pivot com valores explícitos (melhor performance)
receita_cat_ano_otimizado = (
    od_receita
    .join(orders, "order_id")
    .join(products, "product_id")
    .join(categories, "category_id")
    .withColumn("ano", F.year("order_date"))
    .groupBy("category_name")
    .pivot("ano", [1996, 1997, 1998])
    .agg(F.round(F.sum("receita"), 2))
    .orderBy("category_name")
)

receita_cat_ano_otimizado.show(truncate=False)

### Pivot: receita por país (top 10) pivotada por ano

In [ ]:
# Descobrir os top 10 países por receita total
top_paises = [
    row.ship_country for row in
    od_receita
    .join(orders, "order_id")
    .groupBy("ship_country")
    .agg(F.sum("receita").alias("total"))
    .orderBy(F.col("total").desc())
    .limit(10)
    .select("ship_country")
    .collect()
]

print("Top 10 países:", top_paises)

# Pivot com filtro nos top 10 países
receita_pais_ano = (
    od_receita
    .join(orders, "order_id")
    .filter(F.col("ship_country").isin(top_paises))
    .withColumn("ano", F.year("order_date"))
    .groupBy("ship_country")
    .pivot("ano")
    .agg(F.round(F.sum("receita"), 2))
    .orderBy("ship_country")
)

receita_pais_ano.show(truncate=False)

---

## Resumo das Window Functions

| Função | Categoria | Uso | Exemplo Northwind |
|---|---|---|---|
| `row_number()` | Ranking | Numeração sequencial | Ranking de clientes |
| `rank()` | Ranking | Com saltos em empates | Ranking de produtos |
| `dense_rank()` | Ranking | Sem saltos em empates | Ranking denso |
| `lag(col, n)` | Analítica | Valor anterior | Receita do mês anterior |
| `lead(col, n)` | Analítica | Valor seguinte | Previsão próximo mês |
| `sum().over()` | Agregação | Soma acumulada | Receita YTD |
| `avg().over()` | Agregação | Média móvel | Média móvel de vendas |
| `ntile(n)` | Segmentação | Dividir em n grupos | Segmentar clientes |

**Lembre-se:** A chave é definir bem a janela com `Window.partitionBy().orderBy()`.

---

## 6. UDFs — User Defined Functions (Prova)

Quando nao existe funcao built-in para o que voce precisa, crie uma **UDF** (User Defined Function).

- UDFs sao funcoes Python registradas no Spark
- Sao mais **lentas** que funcoes built-in (serialização Python ↔ JVM)
- Use apenas quando nao houver alternativa com `pyspark.sql.functions`

> **Na prova:** Questao 9 — UDF **sem state** = cada linha e processada independentemente, sem depender de linhas anteriores. Ideal para validacoes/transformacoes customizadas.

In [ ]:
from pyspark.sql.types import StringType

# Definir uma função Python normal
def classificar_receita(valor):
    """Classifica receita em faixas — lógica customizada que não existe built-in."""
    if valor is None:
        return "Sem dados"
    elif valor >= 10000:
        return "Premium"
    elif valor >= 5000:
        return "Alto"
    elif valor >= 1000:
        return "Medio"
    else:
        return "Baixo"

# Registrar como UDF do Spark
classificar_udf = F.udf(classificar_receita, StringType())

# Aplicar a UDF no DataFrame de receita por cliente
receita_cliente_udf = (
    od_receita
    .join(orders, "order_id")
    .join(customers, "customer_id")
    .groupBy("company_name")
    .agg(F.round(F.sum("receita"), 2).alias("total"))
    .withColumn("classificacao", classificar_udf(F.col("total")))
    .orderBy(F.col("total").desc())
)

receita_cliente_udf.show(15, truncate=False)

In [ ]:
# UDF registrada para uso em spark.sql() também
spark.udf.register("classificar_receita_sql", classificar_receita, StringType())

# Agora pode usar em SQL puro
receita_cliente.createOrReplaceTempView("receita_view")
spark.sql("""
    SELECT company_name, total, classificar_receita_sql(total) AS classificacao
    FROM receita_view
    ORDER BY total DESC
    LIMIT 10
""").show(truncate=False)

---

## Exercícios

### Exercício 1: Reconstruir `view_receitas_acumuladas`

Reconstrua do zero o Relatório 5 — sem olhar o código acima:
1. Calcule a receita mensal (agrupando por ano e mês)
2. Adicione a coluna `receita_ytd` com soma acumulada no ano
3. Adicione a coluna `crescimento_mensal` com a diferença para o mês anterior
4. Adicione a coluna `percentual_mudanca` com o crescimento percentual

In [ ]:
# Exercício 1 — Sua solução aqui
# Dica: você vai precisar de Window.partitionBy("ano").orderBy("mes")
# e das funções F.sum().over(), F.lag()

### Exercício 2: Segmentação com NTILE

1. Calcule a receita total por cliente
2. Use `ntile(5)` para segmentar em 5 grupos
3. Filtre apenas os clientes de **baixo valor** (grupos 4 e 5)
4. Mostre o resultado ordenado por receita decrescente

In [ ]:
# Exercício 2 — Sua solução aqui
# Dica: Window.orderBy(F.col("total").desc()) + F.ntile(5) + filter

### Exercício 3: Tabela Pivotada — Receita por País x Ano

1. Junte `od_receita` com `orders` para obter `ship_country` e `order_date`
2. Identifique os **top 10 países** por receita total
3. Filtre apenas esses países
4. Crie uma tabela pivotada: linhas = país, colunas = ano, valores = receita total

In [ ]:
# Exercício 3 — Sua solução aqui
# Dica: primeiro calcule os top 10 países, depois use .pivot("ano")